In [1]:
"""
Benchmark: Pandas vs Dask na operacao mean()
=============================================

Objetivo
--------
Medir experimentalmente a partir de qual tamanho de dado o paralelismo do Dask
compensa o custo fixo de construir e escalonar o grafo de tarefas.

Contexto de engenharia
----------------------
O dado sintetico simula uma aquisicao experimental de temperatura: um sinal com
uma tendencia lenta somada a ruido gaussiano do sensor. Calcular a media de uma
serie longa de aquisicao e uma das operacoes mais comuns em pos-processamento
de ensaios (bancada de motores, tunel de vento, ensaios de vibracao).

Aluno: <seu nome>
Disciplina: Computacao de Alto Desempenho em Python
"""

import platform
import time

import dask
import dask.dataframe as dd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# 1. PARAMETROS DO EXPERIMENTO
# ----------------------------------------------------------------------------

# Tamanhos varridos em escala aproximadamente logaritmica.
# 5e7 linhas em float64 = 400 MB. Ajuste o teto conforme a RAM disponivel.
TAMANHOS = [
    100_000,
    300_000,
    1_000_000,
    3_000_000,
    10_000_000,
    30_000_000,
    50_000_000,
]

REPETICOES = 7        # numero de medicoes validas por ponto
AQUECIMENTOS = 2      # execucoes descartadas (cache frio, import tardio)
BYTES_POR_ELEMENTO = 8  # float64

# Largura de banda de memoria estimada da maquina, em GB/s.
# Serve apenas para tracar o piso teorico no grafico.
# Snapdragon X Elite: pico nominal ~135 GB/s; use ~60-70% como valor efetivo.
BANDA_EFETIVA_GB_S = 90.0


# ----------------------------------------------------------------------------
# 2. FUNCAO GENERICA DE MEDICAO
# ----------------------------------------------------------------------------

def medir_tempo(funcao, repeticoes=REPETICOES, aquecimentos=AQUECIMENTOS):
    """
    Executa 'funcao' varias vezes e devolve estatisticas do tempo de execucao.

    Usa perf_counter (relogio monotonico de alta resolucao) e reporta a MEDIANA,
    que e mais robusta a interferencia do sistema operacional do que a media.
    """
    for _ in range(aquecimentos):
        funcao()

    tempos = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        funcao()
        tempos.append(time.perf_counter() - inicio)

    tempos = np.array(tempos)
    return {
        "mediana": float(np.median(tempos)),
        "minimo": float(tempos.min()),
        "desvio": float(tempos.std()),
    }


# ----------------------------------------------------------------------------
# 3. GERACAO DOS DADOS SINTETICOS
# ----------------------------------------------------------------------------

def gerar_dados_sensor(n_amostras, semente=42):
    """
    Gera uma serie de leituras de um sensor de temperatura.

    Modelo: T(t) = T0 + deriva_lenta(t) + ruido_branco
    """
    gerador = np.random.default_rng(semente)

    tempo_s = np.arange(n_amostras, dtype=np.float64)
    deriva_lenta = 5.0 * np.sin(2.0 * np.pi * tempo_s / n_amostras)
    ruido_sensor = gerador.normal(loc=0.0, scale=0.3, size=n_amostras)
    temperatura = 25.0 + deriva_lenta + ruido_sensor

    return pd.DataFrame({"temperatura_C": temperatura})


def escolher_n_particoes(n_amostras, alvo_mb=128):
    """
    Escolhe o numero de particoes do Dask mirando ~128 MB por particao,
    que e a recomendacao pratica da documentacao.

    Particao muito pequena -> overhead de escalonamento domina.
    Particao muito grande  -> pouca oportunidade de paralelismo.
    """
    tamanho_mb = n_amostras * BYTES_POR_ELEMENTO / 1e6
    return max(1, int(round(tamanho_mb / alvo_mb)))


# ----------------------------------------------------------------------------
# 4. VARREDURA PRINCIPAL
# ----------------------------------------------------------------------------

def rodar_benchmark(tamanhos=TAMANHOS):
    """
    Para cada tamanho, mede tres cenarios:
      (a) pandas puro
      (b) dask: apenas o .compute() (DataFrame ja particionado)
      (c) dask: conversao from_pandas + .compute()
    """
    resultados = []

    for n_amostras in tamanhos:
        tamanho_mb = n_amostras * BYTES_POR_ELEMENTO / 1e6
        n_particoes = escolher_n_particoes(n_amostras)

        print(f"N = {n_amostras:>12,}  ({tamanho_mb:7.1f} MB, "
              f"{n_particoes} particoes) ... ", end="", flush=True)

        df_pandas = gerar_dados_sensor(n_amostras)
        df_dask = dd.from_pandas(df_pandas, npartitions=n_particoes)

        # (a) pandas puro
        t_pandas = medir_tempo(
            lambda: df_pandas["temperatura_C"].mean()
        )

        # (b) dask, apenas a execucao do grafo
        t_dask_compute = medir_tempo(
            lambda: df_dask["temperatura_C"].mean().compute()
        )

        # (c) dask, incluindo o custo de particionar o dado
        t_dask_total = medir_tempo(
            lambda: dd.from_pandas(
                df_pandas, npartitions=n_particoes
            )["temperatura_C"].mean().compute(),
            repeticoes=3,      # este cenario e mais caro; menos repeticoes
            aquecimentos=1,
        )

        # Verificacao de corretude: os tres caminhos devem dar o mesmo numero
        media_ref = df_pandas["temperatura_C"].mean()
        media_dask = df_dask["temperatura_C"].mean().compute()
        assert np.isclose(media_ref, media_dask, rtol=1e-10), \
            "Divergencia entre pandas e dask!"

        resultados.append({
            "n_amostras": n_amostras,
            "tamanho_mb": tamanho_mb,
            "n_particoes": n_particoes,
            "t_pandas": t_pandas["mediana"],
            "t_dask_compute": t_dask_compute["mediana"],
            "t_dask_total": t_dask_total["mediana"],
            "desvio_pandas": t_pandas["desvio"],
            "desvio_dask": t_dask_compute["desvio"],
        })

        print(f"pandas {t_pandas['mediana']*1e3:8.2f} ms | "
              f"dask {t_dask_compute['mediana']*1e3:8.2f} ms")

        # Libera memoria antes do proximo tamanho
        del df_pandas, df_dask

    tabela = pd.DataFrame(resultados)
    tabela["speedup"] = tabela["t_pandas"] / tabela["t_dask_compute"]
    tabela["speedup_total"] = tabela["t_pandas"] / tabela["t_dask_total"]
    return tabela


# ----------------------------------------------------------------------------
# 5. GRAFICOS
# ----------------------------------------------------------------------------

def plotar_resultados(tabela, arquivo_saida="benchmark_pandas_dask.png"):
    figura, (eixo_tempo, eixo_speedup) = plt.subplots(1, 2, figsize=(13, 5.2))

    n = tabela["n_amostras"]

    # ---- Painel esquerdo: tempo absoluto (log-log) ----
    eixo_tempo.loglog(n, tabela["t_pandas"] * 1e3,
                      "o-", label="Pandas", linewidth=2)
    eixo_tempo.loglog(n, tabela["t_dask_compute"] * 1e3,
                      "s-", label="Dask (apenas compute)", linewidth=2)
    eixo_tempo.loglog(n, tabela["t_dask_total"] * 1e3,
                      "^--", label="Dask (from_pandas + compute)",
                      linewidth=1.5, alpha=0.8)

    # Piso teorico imposto pela largura de banda de memoria
    tempo_banda_ms = (n * BYTES_POR_ELEMENTO / (BANDA_EFETIVA_GB_S * 1e9)) * 1e3
    eixo_tempo.loglog(n, tempo_banda_ms, ":", color="gray",
                      label=f"Limite de banda (~{BANDA_EFETIVA_GB_S:.0f} GB/s)")

    eixo_tempo.set_xlabel("Numero de amostras N")
    eixo_tempo.set_ylabel("Tempo de execucao [ms]")
    eixo_tempo.set_title("Custo da operacao mean()")
    eixo_tempo.grid(True, which="both", alpha=0.3)
    eixo_tempo.legend(fontsize=9)

    # ---- Painel direito: speedup ----
    eixo_speedup.semilogx(n, tabela["speedup"],
                          "s-", color="tab:orange", linewidth=2,
                          label="Dask (apenas compute)")
    eixo_speedup.semilogx(n, tabela["speedup_total"],
                          "^--", color="tab:green", linewidth=1.5,
                          label="Dask (from_pandas + compute)")
    eixo_speedup.axhline(1.0, color="black", linestyle="--", linewidth=1.2)
    eixo_speedup.text(n.iloc[0], 1.05, "break-even (S = 1)", fontsize=9)

    eixo_speedup.set_xlabel("Numero de amostras N")
    eixo_speedup.set_ylabel("Speedup  $S = T_{pandas} / T_{dask}$")
    eixo_speedup.set_title("Ganho relativo do Dask")
    eixo_speedup.grid(True, which="both", alpha=0.3)
    eixo_speedup.legend(fontsize=9)

    figura.suptitle(
        "Pandas vs Dask - reducao mean() sobre serie de aquisicao",
        fontsize=13
    )
    figura.tight_layout()
    figura.savefig(arquivo_saida, dpi=150)
    print(f"\nGrafico salvo em: {arquivo_saida}")
    plt.show()


# ----------------------------------------------------------------------------
# 6. EXECUCAO
# ----------------------------------------------------------------------------

if __name__ == "__main__":
    print("=" * 70)
    print("AMBIENTE")
    print("=" * 70)
    print(f"Plataforma  : {platform.platform()}")
    print(f"Arquitetura : {platform.machine()}   "
          f"(esperado ARM64 no Snapdragon X Elite)")
    print(f"Processador : {platform.processor()}")
    print(f"Python      : {platform.python_version()}")
    print(f"NumPy       : {np.__version__}")
    print(f"Pandas      : {pd.__version__}")
    print(f"Dask        : {dask.__version__}")
    print()

    tabela_resultados = rodar_benchmark()

    print("\n" + "=" * 70)
    print("RESULTADOS")
    print("=" * 70)
    colunas = ["n_amostras", "tamanho_mb", "n_particoes",
               "t_pandas", "t_dask_compute", "speedup"]
    print(tabela_resultados[colunas].to_string(index=False,
                                               float_format="%.5f"))

    tabela_resultados.to_csv("resultados_benchmark.csv", index=False)
    plotar_resultados(tabela_resultados)

AMBIENTE
Plataforma  : Windows-11-10.0.26200-SP0
Arquitetura : ARM64   (esperado ARM64 no Snapdragon X Elite)
Processador : ARMv8 (64-bit) Family 8 Model 1 Revision 201, Qualcomm Technologies Inc
Python      : 3.13.9
NumPy       : 2.3.5
Pandas      : 2.3.3
Dask        : unknown

N =      100,000  (    0.8 MB, 1 particoes) ... pandas     0.16 ms | dask     7.61 ms
N =      300,000  (    2.4 MB, 1 particoes) ... pandas     0.90 ms | dask    12.00 ms
N =    1,000,000  (    8.0 MB, 1 particoes) ... pandas     2.87 ms | dask    11.29 ms
N =    3,000,000  (   24.0 MB, 1 particoes) ... pandas    10.06 ms | dask    21.05 ms
N =   10,000,000  (   80.0 MB, 1 particoes) ... pandas    32.10 ms | dask    42.22 ms
N =   30,000,000  (  240.0 MB, 2 particoes) ... 

KeyboardInterrupt: 